In [ ]:

!pip -q install numpy pandas scikit-learn catboost optuna tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 22.3 MB/s eta 0:00:00


In [ ]:

import subprocess

subprocess.run(["nvidia-smi"], check=False)

from catboost.utils import get_gpu_device_count
gpu_count = int(get_gpu_device_count())
print("CatBoost GPU devices:", gpu_count)
CB_TASK = "GPU" if gpu_count > 0 else "CPU"
print("Using task_type=", CB_TASK)


CatBoost GPU devices: 1
Using task_type= GPU


In [11]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def macro_f1(y_true: np.ndarray, y_pred: np.ndarray, *, num_classes: int) -> float:
    from sklearn.metrics import f1_score

    labels = list(range(num_classes))
    return float(f1_score(y_true, y_pred, labels=labels, average="macro"))


@dataclass(frozen=True)
class Splitter:
    cv: object
    groups: np.ndarray | None

    def split(self, x: pd.DataFrame, y: np.ndarray):
        if self.groups is None:
            yield from self.cv.split(x, y)
        else:
            yield from self.cv.split(x, y, groups=self.groups)


def make_splitter(*, cv: str, n_splits: int, seed: int, groups_df: pd.DataFrame | None) -> Splitter:
    if cv == "stratified":
        from sklearn.model_selection import StratifiedKFold

        return Splitter(
            cv=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed),
            groups=None,
        )

    if cv == "group_ab":
        if groups_df is None:
            raise ValueError("groups_df is required for cv='group_ab'")
        from sklearn.model_selection import GroupKFold

        groups = (groups_df["Group_A"].astype("int64") * 1_000_000) + groups_df["Group_B"].astype("int64")
        return Splitter(cv=GroupKFold(n_splits=n_splits), groups=groups.to_numpy())

    raise ValueError(f"Unknown cv={cv!r}")


ID_COL = "Node_ID"
TARGET_COL = "Label_Target"


def _encode_labels(y_raw: pd.Series) -> tuple[np.ndarray, list[int]]:
    y_num = pd.to_numeric(y_raw, errors="coerce")
    if y_num.isna().any():
        bad = int(y_num.isna().sum())
        raise ValueError(f"{TARGET_COL} has {bad} non-numeric/missing values")

    y_int = y_num.astype(np.int64)
    label_values = sorted(pd.unique(y_int).tolist())
    if len(label_values) != 7:
        raise ValueError(
            f"{TARGET_COL} must contain exactly 7 classes; got {len(label_values)}: {label_values[:20]}"
        )
    mapping = {v: i for i, v in enumerate(label_values)}
    y_idx = y_int.map(mapping).to_numpy(dtype=np.int64)
    return y_idx, [int(v) for v in label_values]


def _safe_log1p(x: pd.Series) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    return pd.Series(np.log1p(np.clip(x.to_numpy(dtype=np.float64), 0.0, None)), index=x.index)


def build_features(
    df: pd.DataFrame, *, is_train: bool
) -> tuple[pd.DataFrame, np.ndarray | None, np.ndarray, list[str], list[str], list[int] | None]:
    if ID_COL not in df.columns:
        raise ValueError(f"Missing required id column: {ID_COL}")

    y = None
    label_values: list[int] | None = None
    if is_train:
        if TARGET_COL not in df.columns:
            raise ValueError(f"Missing required target column: {TARGET_COL}")
        y, label_values = _encode_labels(df[TARGET_COL])

    id_values = df[ID_COL].astype(str).to_numpy()
    x = df.drop(columns=[c for c in [ID_COL, TARGET_COL] if c in df.columns]).copy()

    expected_cat = ["Group_A", "Group_B"]
    cat_features = [c for c in expected_cat if c in x.columns]

    if "Attr_02" in x.columns:
        angle_rad = np.deg2rad(pd.to_numeric(x["Attr_02"], errors="coerce").to_numpy())
        x["Attr_02_sin"] = np.sin(angle_rad)
        x["Attr_02_cos"] = np.cos(angle_rad)

    log_cols = [
        "Attr_04",
        "Attr_06",
        "Engineered_Dist_H",
        "Engineered_Density",
        "Engineered_Log_Dist",
    ]
    for c in log_cols:
        if c in x.columns:
            x[f"log1p_{c}"] = _safe_log1p(x[c])

    if {"Attr_01", "Attr_03"}.issubset(x.columns):
        a1 = pd.to_numeric(x["Attr_01"], errors="coerce")
        a3 = pd.to_numeric(x["Attr_03"], errors="coerce")
        x["Attr_01_x_Attr_03"] = (a1 * a3).to_numpy()
        x["Attr_03_sq"] = (a3 * a3).to_numpy()

    if {"Engineered_Flow_X", "Engineered_Momentum"}.issubset(x.columns):
        fx = pd.to_numeric(x["Engineered_Flow_X"], errors="coerce")
        mo = pd.to_numeric(x["Engineered_Momentum"], errors="coerce")
        x["FlowX_x_Momentum"] = (fx * mo).to_numpy()
        x["abs_Momentum"] = np.abs(mo.to_numpy(dtype=np.float64))

    if {"Engineered_Dist_H", "Attr_04"}.issubset(x.columns):
        dh = pd.to_numeric(x["Engineered_Dist_H"], errors="coerce").to_numpy(dtype=np.float64)
        d4 = pd.to_numeric(x["Attr_04"], errors="coerce").to_numpy(dtype=np.float64)
        x["DistH_over_Attr04"] = dh / (np.abs(d4) + 1.0)

    for c in x.columns:
        if c in cat_features:
            continue
        x[c] = pd.to_numeric(x[c], errors="coerce").astype(np.float32)

    for c in list(x.columns):
        if c in cat_features:
            continue
        x[f"isna_{c}"] = x[c].isna().astype(np.int8)

    for c in cat_features:
        x[c] = pd.to_numeric(x[c], errors="coerce").astype("Int64").astype("category")

    feature_names = list(x.columns)
    return x, y, id_values, feature_names, cat_features, label_values


def train_fold_catboost(
    *,
    x_tr: pd.DataFrame,
    y_tr: np.ndarray,
    x_va: pd.DataFrame,
    y_va: np.ndarray,
    x_test: pd.DataFrame,
    categorical_features: list[str],
    seed: int,
    task_type: str = "GPU",
    iterations: int = 20_000,
    learning_rate: float = 0.03,
    depth: int = 8,
    l2_leaf_reg: float = 3.0,
    subsample: float = 0.8,
    colsample_bylevel: float = 0.8,
    early_stopping_rounds: int = 500,
) -> tuple[object, np.ndarray, np.ndarray]:
    from catboost import CatBoostClassifier
    from sklearn.utils.class_weight import compute_class_weight

    cat_features_idx = [int(x_tr.columns.get_loc(c)) for c in categorical_features]

    classes = np.unique(y_tr)
    class_w = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
    class_weights = {int(c): float(w) for c, w in zip(classes, class_w)}

    params: dict[str, Any] = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": int(iterations),
        "learning_rate": float(learning_rate),
        "depth": int(depth),
        "l2_leaf_reg": float(l2_leaf_reg),
        "subsample": float(subsample),
        "colsample_bylevel": float(colsample_bylevel),
        "random_seed": int(seed),
        "allow_writing_files": False,
        "class_weights": class_weights,
        "od_type": "Iter",
        "od_wait": int(early_stopping_rounds),
        "verbose": 250,
        "task_type": str(task_type).upper(),
        "bootstrap_type": "Bernoulli",  # Added to support 'subsample'
    }

    try:
        model = CatBoostClassifier(**params)
        model.fit(
            x_tr,
            y_tr,
            cat_features=cat_features_idx,
            eval_set=(x_va, y_va),
            use_best_model=True,
        )
    except Exception as e:
        if str(task_type).upper() == "GPU":
            print(
                "CatBoost GPU training failed; retrying on CPU.\n"
                f"Reason: {type(e).__name__}: {e}"
            )
            params["task_type"] = "CPU"
            model = CatBoostClassifier(**params)
            model.fit(
                x_tr,
                y_tr,
                cat_features=cat_features_idx,
                eval_set=(x_va, y_va),
                use_best_model=True,
            )
        else:
            raise

    va_proba = model.predict_proba(x_va)
    te_proba = model.predict_proba(x_test)
    return model, va_proba, te_proba

In [6]:
# Load data
DATA_DIR = Path("data")
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)



In [8]:
train_df

,Node_ID,Attr_01,Attr_02,Attr_03,Attr_04,Attr_05,Attr_06,Group_A,Group_B,Engineered_Dist_H,Engineered_Flow_X,Engineered_Density,Engineered_Momentum,Engineered_Stress_V,Engineered_Log_Dist,Label_Target
0,6dcee7abc4,3102.004066,359.006060,18.999961,1370.778455,201.998393,391.016152,1,24,2.999431,182.097269,3457.870449,880.885817,41.004661,62.038421,7.0
1,65e87e5b91,2228.294676,52.006525,16.000854,617.910338,204.008756,89.997142,6,2,4.000086,318.500947,4800.206679,353.993292,121.003434,37.875125,6.0
2,92013fe0c4,3232.965858,267.030029,17.002805,6182.816328,244.959257,1870.836753,2,30,1.000158,719.968719,11897.077485,4027.006406,-32.996332,58.200006,8.0
3,931599fde5,2716.020436,164.982226,28.999586,1395.923074,233.982721,1681.180184,2,13,2.999538,62.636281,1739.942772,1538.403354,118.017120,81.469319,7.0
4,720fe115a3,2981.319803,31.999905,8.001329,1994.033169,224.014007,3925.306155,1,23,1.000030,42.099314,335.935988,2959.838059,77.998821,26.827524,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127773,37bd9854c0,2945.052328,43.998973,24.999552,1972.331931,176.026979,1567.097246,1,33,3.000033,199.724176,4800.820685,1769.531861,134.993241,76.574776,7.0
127774,034d573f21,3059.843965,230.999995,9.999674,5977.116495,249.009314,2789.258217,2,29,0.999830,382.819975,3819.806505,4382.985834,21.993717,33.660796,8.0
127775,1d1178ea09,3113.669731,85.010456,6.000002,4401.281777,229.988819,1624.340508,1,22,1.000044,300.616138,1800.054458,3012.482402,92.002171,21.795422,8.0
127776,634706fbb9,2706.200364,191.007426,29.003442,2470.926880,247.033038,815.971459,2,4,3.000139,130.389425,3479.500854,1643.328109,46.004539,81.191850,7.0


In [9]:
train_df.dropna(subset=[TARGET_COL], inplace=True)
x_train, y_train, id_train, feature_names, cat_features, label_values = build_features(train_df, is_train=True)
x_test, _, id_test, _, _, _ = build_features(test_df, is_train=False)

num_classes = int(len(label_values))
print("train:", x_train.shape, "test:", x_test.shape)
print("categorical:", cat_features)
print("label_values:", label_values)
print("num_classes:", num_classes)

train: (127777, 50) test: (102202, 50)
categorical: ['Group_A', 'Group_B']
label_values: [0, 3, 4, 5, 6, 7, 8]
num_classes: 7


In [ ]:
import optuna
from tqdm.auto import tqdm

SEED = 42
N_SPLITS_TUNE = 3

splitter = make_splitter(cv="stratified", n_splits=N_SPLITS_TUNE, seed=SEED, groups_df=None)

def objective(trial: optuna.Trial) -> float:
    params = {
        "iterations": trial.suggest_int("iterations", 2000, 20000, step=1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "depth": trial.suggest_int("depth", 6, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.6, 1.0),
    }


    if CB_TASK == "GPU":
        params["subsample"] = 1.0
        params["colsample_bylevel"] = 1.0

    oof_pred = np.full_like(y_train, fill_value=-1)
    for fold, (tr_idx, va_idx) in enumerate(splitter.split(x_train, y_train)):
        x_tr, y_tr = x_train.iloc[tr_idx], y_train[tr_idx]
        x_va, y_va = x_train.iloc[va_idx], y_train[va_idx]

        _, va_proba, _ = train_fold_catboost(
            x_tr=x_tr,
            y_tr=y_tr,
            x_va=x_va,
            y_va=y_va,
            x_test=x_test.iloc[:1], 
            categorical_features=cat_features,
            seed=SEED + fold,
            task_type=CB_TASK,
            iterations=params["iterations"],
            learning_rate=params["learning_rate"],
            depth=params["depth"],
            l2_leaf_reg=params["l2_leaf_reg"],
            subsample=params["subsample"],
            colsample_bylevel=params["colsample_bylevel"],
            early_stopping_rounds=500,
        )

        oof_pred[va_idx] = va_proba.argmax(axis=1)

    score = macro_f1(y_train, oof_pred, num_classes=num_classes)
    return float(score)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=25, show_progress_bar=True)

print("best value:", study.best_value)
print("best params:", study.best_params)


[I 2026-03-04 17:02:50,018] A new study created in memory with name: no-name-f0962fd9-ff41-476d-9e22-9a0c06f2dccb


  0%|          | 0/25 [00:00<?, ?it/s]

0:	learn: 1.3937882	test: 1.4233069	best: 1.4233069 (0)	total: 68.6ms	remaining: 10m 17s
250:	learn: 0.0084080	test: 0.8903892	best: 0.5086075 (22)	total: 10.3s	remaining: 5m 59s
500:	learn: 0.0040610	test: 1.0412211	best: 0.5086075 (22)	total: 19.5s	remaining: 5m 30s
bestTest = 0.508607453
bestIteration = 22
Shrink model to first 23 iterations.
0:	learn: 1.4054044	test: 1.4558356	best: 1.4558356 (0)	total: 32ms	remaining: 4m 47s
250:	learn: 0.0093835	test: 0.6009161	best: 0.3976671 (27)	total: 8.44s	remaining: 4m 54s
500:	learn: 0.0042171	test: 0.6921726	best: 0.3976671 (27)	total: 18.5s	remaining: 5m 14s
bestTest = 0.3976670753
bestIteration = 27
Shrink model to first 28 iterations.
0:	learn: 1.3891283	test: 1.4449177	best: 1.4449177 (0)	total: 31.5ms	remaining: 4m 43s
250:	learn: 0.0089755	test: 0.8769596	best: 0.5080957 (22)	total: 10.1s	remaining: 5m 51s
500:	learn: 0.0051596	test: 0.9400405	best: 0.5080957 (22)	total: 20.6s	remaining: 5m 49s
bestTest = 0.5080957353
bestIteration 

In [ ]:
# Train full CV with best params and write submission
import json

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

best = study.best_params.copy() 


if CB_TASK == "GPU":
    best["subsample"] = 1.0
    best["colsample_bylevel"] = 1.0

N_SPLITS = 5
splitter_full = make_splitter(cv="stratified", n_splits=N_SPLITS, seed=SEED, groups_df=None)

oof_pred = np.full_like(y_train, fill_value=-1)
test_proba = np.zeros((x_test.shape[0], num_classes), dtype=np.float64)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(splitter_full.split(x_train, y_train)):
    x_tr, y_tr = x_train.iloc[tr_idx], y_train[tr_idx]
    x_va, y_va = x_train.iloc[va_idx], y_train[va_idx]

    model, va_proba, te_proba = train_fold_catboost(
        x_tr=x_tr,
        y_tr=y_tr,
        x_va=x_va,
        y_va=y_va,
        x_test=x_test,
        categorical_features=cat_features,
        seed=SEED + fold,
        task_type=CB_TASK,
        iterations=int(best["iterations"]),
        learning_rate=float(best["learning_rate"]),
        depth=int(best["depth"]),
        l2_leaf_reg=float(best["l2_leaf_reg"]),
        subsample=float(best["subsample"]),
        colsample_bylevel=float(best["colsample_bylevel"]),
        early_stopping_rounds=500,
    )

    model.save_model(str(ARTIFACTS_DIR / f"cb_fold{fold}.cbm"))

    oof_pred[va_idx] = va_proba.argmax(axis=1)
    test_proba += te_proba / N_SPLITS

    fold_f1 = macro_f1(y_va, va_proba.argmax(axis=1), num_classes=num_classes)
    fold_scores.append(float(fold_f1))
    print(f"[fold {fold}] macro_f1={fold_f1:.6f}")

oof_score = macro_f1(y_train, oof_pred, num_classes=num_classes)
print(f"OOF macro_f1={oof_score:.6f} (fold mean={np.mean(fold_scores):.6f})")

test_pred = np.take(np.asarray(label_values, dtype=np.int64), test_proba.argmax(axis=1))
submission = pd.DataFrame({"Node_ID": id_test, "Label_Target_Predicted": test_pred.astype(np.int64)})
submission.to_csv(ARTIFACTS_DIR / "submission.csv", index=False)
submission.to_csv("submission.csv", index=False)

meta = {
    "model": "catboost",
    "cv": "stratified",
    "n_splits": N_SPLITS,
    "seed": SEED,
    "num_classes": num_classes,
    "label_values": label_values,
    "feature_names": feature_names,
    "categorical_features": cat_features,
    "cb_task": CB_TASK,
    "best_params": best,
    "oof_macro_f1": float(oof_score),
    "fold_macro_f1": [float(s) for s in fold_scores],
}
with (ARTIFACTS_DIR / "meta.json").open("w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Wrote:", ARTIFACTS_DIR / "submission.csv")
print("Wrote:", ARTIFACTS_DIR / "meta.json")

0:	learn: 1.6971528	test: 1.7153777	best: 1.7153777 (0)	total: 119ms	remaining: 35m 39s
250:	learn: 0.0242228	test: 0.5060464	best: 0.3828642 (90)	total: 16.3s	remaining: 19m 14s
500:	learn: 0.0101283	test: 0.6154309	best: 0.3828642 (90)	total: 32.2s	remaining: 18m 44s
bestTest = 0.3828641837
bestIteration = 90
Shrink model to first 91 iterations.
[fold 0] macro_f1=0.786778
0:	learn: 1.7009723	test: 1.7190940	best: 1.7190940 (0)	total: 55.3ms	remaining: 16m 35s
250:	learn: 0.0235481	test: 0.6969597	best: 0.4745959 (54)	total: 15.6s	remaining: 18m 26s
500:	learn: 0.0101818	test: 0.8569446	best: 0.4745959 (54)	total: 32.2s	remaining: 18m 43s
bestTest = 0.4745959047
bestIteration = 54
Shrink model to first 55 iterations.
[fold 1] macro_f1=0.781472
0:	learn: 1.6940353	test: 1.7145020	best: 1.7145020 (0)	total: 54.1ms	remaining: 16m 14s
250:	learn: 0.0242195	test: 0.4993437	best: 0.4224572 (63)	total: 15.6s	remaining: 18m 22s
500:	learn: 0.0106784	test: 0.5936712	best: 0.4224572 (63)	total: